In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versioni dei package">

Il codice di questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Puoi usare le opzioni per personalizzare la primitiva Estimator. Sebbene l'interfaccia del metodo `run()` delle primitive sia comune a tutte le implementazioni, le loro opzioni non lo sono. Consulta i riferimenti API per informazioni sulle opzioni di [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) e [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## Impostare le opzioni di Estimator

Puoi impostare le opzioni durante l'inizializzazione di Estimator, dopo l'inizializzazione di Estimator, oppure puoi aggiornare le opzioni dopo che Estimator è stato inizializzato. Per istruzioni su come usare queste tecniche, consulta il topic [Introduzione alle opzioni](/guides/runtime-options-overview#options-precedence).

Inoltre, puoi impostare il valore `precision` nel metodo `run()`, come descritto nella sezione seguente.
<span id="run-method"></span>

### Metodo Run()

Gli unici valori che puoi passare a `run()` sono quelli definiti nell'interfaccia. Ovvero `precision` per Estimator. Questo sovrascrive qualsiasi valore impostato per `default_precision` per l'esecuzione corrente.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Caso speciale: precision
Il metodo `EstimatorV2.run` accetta due argomenti: una lista di PUB, ognuno dei quali può specificare un valore di precision specifico per PUB, e un argomento keyword precision. Questi valori di precision fanno parte dell'interfaccia di esecuzione di Estimator, e sono indipendenti dalle opzioni di Runtime Estimator. Hanno la precedenza su qualsiasi valore specificato come opzione, al fine di rispettare l'astrazione di Estimator.

Tuttavia, se `precision` non è specificato da alcun PUB né nell'argomento keyword run (o se sono tutti `None`), viene usato il valore di precision dalle opzioni, in particolare `default_precision`.

Nota che le opzioni di Estimator contengono sia `default_shots` che `default_precision`. Tuttavia, poiché gate-twirling è abilitato per impostazione predefinita, il prodotto di `num_randomizations` e `shots_per_randomization` ha la precedenza su queste due opzioni.

Nello specifico, per ogni PUB di Estimator:

1. Se il PUB specifica precision, usa quel valore.
2. Se l'argomento keyword precision è specificato in `run`, usa quel valore.
3. Se `twirling` è abilitato (True per impostazione predefinita), viene usato il prodotto di `num_randomizations` e `shots_per_randomization`, come specificato nelle [opzioni twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
4. Se `estimator.options.default_shots` è specificato, usa quel valore per controllare la quantità di dati.
5. Se `estimator.options.default_precision` è specificato, usa quel valore.

Ad esempio, se precision è specificato in tutti e quattro i posti, viene usato quello con priorità più alta (precision specificato nel PUB).

> **Note:** La precision scala inversamente con l'utilizzo. Ovvero, minore è la precision, più tempo QPU è necessario per eseguire.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## Disattivare tutte le mitigazioni e soppressioni degli errori
Puoi disattivare tutte le mitigazioni e soppressioni degli errori se, ad esempio, stai facendo ricerca sulle tue tecniche di mitigazione. Per farlo, imposta `resilience_level = 0`.

Esempio:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## Opzioni disponibili
La tabella seguente documenta le opzioni dell'ultima versione di `qiskit-ibm-runtime`. Per vedere le versioni di opzioni precedenti, visita il [riferimento API di `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) e seleziona una versione precedente.

<Accordion>
<AccordionItem title="`default_shots`">

Il numero totale di shot da usare per circuito per configurazione.

**Scelte**: Intero >= 0

**Predefinito**: None

[Documentazione API `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

La precision predefinita da usare per qualsiasi PUB o chiamata `run()` che non ne specifica una.

**Scelte**: Float > 0

**Predefinito**: 0.015625 (1 / sqrt(4096))

[Documentazione API `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

Controlla le impostazioni di mitigazione degli errori di dynamical decoupling.

[Documentazione API `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Scelte**: `True`, `False`

**Predefinito**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Scelte**: `middle`, `edges`

**Predefinito**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Scelte: `asap`, `alap`
Predefinito: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

Scelte: `XX`, `XpXm`, `XY4`
Predefinito: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Scelte: `True`, `False`
Predefinito: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[Documentazione API `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Funzione callable che riceve l'`ID del job` e il `risultato del job`.

**Scelte**: None

**Predefinito**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

Lista di tag.

**Scelte**: None

**Predefinito**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**Scelte**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Predefinito**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**Scelte**: `True`, `False`

**Predefinito**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[Documentazione API `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Indica se resettare i qubit allo stato fondamentale per ogni shot.

**Scelte**: `True`, `False`

**Predefinito**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
Il ritardo tra una misurazione e il successivo circuito quantistico.

**Scelte**: Valore nell'intervallo fornito da `backend.rep_delay_range`

**Predefinito**: Dato da `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
Limita per quanto tempo può essere eseguito un job, in secondi. Consulta la guida sul [tempo massimo di esecuzione](/guides/max-execution-time) per i dettagli.

**Scelte**: Numero intero di secondi nell'intervallo [1, 10800]

**Predefinito**: 10800 (3 ore)
  </AccordionItem>

<AccordionItem title="`resilience`">
Opzioni avanzate di resilienza per regolare con precisione la strategia di resilienza.

[Documentazione API `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Opzioni per l'apprendimento del rumore a strati.

[Documentazione API `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Scelte**: list[int] di 2-10 valori nell'intervallo [0, 200]

**Predefinito**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Scelte**: None, Intero >= 1

**Predefinito**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Scelte**: Intero >= 1

**Predefinito**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Scelte**: Intero >= 1

**Predefinito**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**Scelte**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Predefinito**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**Scelte**: `True`, `False`

**Predefinito**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

Opzioni per l'apprendimento del rumore di misurazione.

[Documentazione API `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Scelte**: Intero >= 1

**Predefinito**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Scelte**: Intero, `auto`

**Predefinito**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**Scelte**: `True`, `False`

**Predefinito**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

Opzioni di mitigazione della cancellazione probabilistica degli errori.

[Documentazione API `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**Scelte**: `None`, Intero >= 1

**Predefinito**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**Scelte**: `auto`, float nell'intervallo [0, 1]

**Predefinito**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**Scelte**: `True`, `False`

**Predefinito**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[Documentazione API `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**Scelte**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Predefinito**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Scelte**: Lista di float

**Predefinito**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**Scelte**: Uno o più di: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Predefinito**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**Scelte**: Lista di float; ogni float >= 1

**Predefinito**: `(1, 1.5, 2)` per `PEA`, e `(1, 3, 5)` altrimenti

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

Quanto resilienza costruire contro gli errori. Livelli più alti generano risultati più accurati a scapito di tempi di elaborazione più lunghi. Consulta la sezione [livelli di resilienza](/guides/estimator-noise-management#resilience) nel topic Gestione del rumore per saperne di più.

**Scelte**: `0`, `1`, `2`

**Predefinito**: `1`

[Documentazione API `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**Scelte**: Intero

**Predefinito**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

Opzioni da passare durante la simulazione di un backend

[Documentazione API `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Scelte**: Lista di nomi di gate di base su cui decomporre

**Predefinito**: L'insieme di tutti i gate di base supportati dal [simulatore Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**Scelte**: Lista di interazioni bidirezionali a due qubit

**Predefinito**: None, che implica nessun vincolo di connettività (connettività completa).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**Scelte**: [Qiskit Aer NoiseModel](/guides/build-noise-models), o la sua rappresentazione

**Predefinito**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**Scelte**: Intero

**Predefinito**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Opzioni di twirling

[Documentazione API `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Scelte**: True, False

**Predefinito**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**Scelte**: True, False

**Predefinito**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**Scelte**: `auto`, Intero >= 1

**Predefinito**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**Scelte**: `auto`, Intero >= 1

**Predefinito**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**Scelte**: `active`, `active-circuit`, `active-accum`, `all`

**Predefinito**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

Opzioni sperimentali, quando disponibili.

  </AccordionItem>

</Accordion>
<span id="options-compatibility-table"></span>
## Compatibilità delle funzionalità
Alcune funzionalità di runtime non possono essere usate insieme in un singolo job. Clicca sulla scheda appropriata per un elenco delle funzionalità incompatibili con la funzionalità selezionata:

<Tabs>

  <TabItem value="Fractional" label="Fractional gates">
  Incompatibile con:
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    Incompatibile con:
  - PEA
  - PEC

  Potrebbe non funzionare quando si usano gate personalizzati.
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  Incompatibile con fractional gate o con stretches.

  Altre note:
  - Il measurement twirling può essere applicato solo alle misurazioni terminali.
  - Non funziona con entangler non-Clifford.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    Incompatibile con:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    Incompatibile con:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>
## Prossimi passi
> **Tip:** - Consulta la guida [Introduzione alle opzioni](/guides/runtime-options-overview).
> - Trova ulteriori dettagli sui metodi `EstimatorV2` nel [riferimento API di Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - Decidi in quale [modalità di esecuzione](/guides/execution-modes) eseguire il tuo job.
> - Scopri la [gestione del rumore con Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>